# Chapter 8. 데이터 분석하기

| 섹션 | 내용 |
|------|------|
| 5-1. 상관 분석 | 두 변수 간 선형 관계 강도·방향 측정 (`df.corr`, heatmap, pairplot) |
| 5-2. 회귀 분석 | 독립변수가 종속변수에 미치는 영향 모델링 (단순/다중/로지스틱) |
| 5-3. 가설 검정 | 통계적 유의미성 판단 (t-test, 카이제곱) |
| 5-4. 군집 분석 | 비슷한 데이터끼리 그룹화 (K-Means) |
| 5-5. 주성분 분석 | 고차원 데이터 압축·시각화 (PCA) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import scipy.stats as stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

mpg      = sns.load_dataset('mpg').dropna()
penguins = sns.load_dataset('penguins').dropna()
tips     = sns.load_dataset('tips')
print('mpg:', mpg.shape, '| penguins:', penguins.shape, '| tips:', tips.shape)

## 5-1. 상관 분석 (Correlation Analysis)

**정의**: 두 변수 간의 **선형 관계 강도와 방향**을 -1 ~ 1 사이 숫자로 나타냅니다.

| 상관계수 (r) | 해석 |
|--------------|------|
| 0.9 ~ 1.0 | 매우 강한 양의 상관 |
| 0.7 ~ 0.9 | 강한 양의 상관 |
| 0.4 ~ 0.7 | 중간 양의 상관 |
| 0.1 ~ 0.4 | 약한 양의 상관 |
| -0.1 ~ 0.1 | 거의 무상관 |
| -0.4 ~ -0.1 | 약한 음의 상관 |
| -0.7 ~ -0.4 | 중간 음의 상관 |
| -1.0 ~ -0.7 | 강한 음의 상관 |

| 항목 | 설명 |
|------|------|
| `df.corr()` | Pearson 상관계수 행렬 계산 |
| `df.corr(method='spearman')` | 순위 기반 Spearman 상관계수 |
| `df.corr(method='kendall')` | 순위 일치도 기반 Kendall 상관계수 |

> **주의**: 상관관계 ≠ 인과관계. 아이스크림 판매량과 익사 사고 수는 상관이 높지만 원인이 아님 (공통 원인: 여름)

In [ ]:
# 상관계수 행렬
num_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
corr_matrix = mpg[num_cols].corr()
print(corr_matrix.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히트맵: 상관행렬 시각화
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('MPG 변수 상관행렬 (Heatmap)')

# 특정 변수 쌍 상관 강조
top_corr = corr_matrix['mpg'].drop('mpg').sort_values()
top_corr.plot(kind='barh', ax=axes[1], color=['red' if v < 0 else 'steelblue' for v in top_corr])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('연비(mpg)와 각 변수의 상관계수')

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot: 모든 변수 쌍 관계 한눈에
pair_cols = ['mpg', 'horsepower', 'weight', 'acceleration', 'origin']
sns.pairplot(mpg[pair_cols], hue='origin', diag_kind='kde', height=2.2)
plt.suptitle('MPG 주요 변수 쌍별 산점도 (Pairplot)', y=1.02)
plt.show()

In [ ]:
# 특정 두 변수 간 상관계수와 p-value 확인
r, p = stats.pearsonr(mpg['weight'], mpg['mpg'])
print(f'weight vs mpg')
print(f'  Pearson r = {r:.4f}')
print(f'  p-value   = {p:.4e}')
print(f'  해석: {"통계적으로 유의미한" if p < 0.05 else "유의미하지 않은"} 상관관계')

## 5-2. 회귀 분석 (Regression Analysis)

**정의**: 독립변수(원인)가 종속변수(결과)에 미치는 **영향을 수식으로 모델링**하고 예측합니다.

**핵심 질문**: *"A가 변하면 B가 얼마나, 어떻게 변할까?"*

| 유형 | 독립변수 수 | 종속변수 | 사용 상황 |
|------|------------|----------|----------|
| 단순 선형 회귀 | 1개 | 연속형 | 공부시간 → 시험점수 |
| 다중 선형 회귀 | 2개 이상 | 연속형 | 마력+무게+연식 → 연비 |
| 로지스틱 회귀 | 1개 이상 | 범주형(0/1) | 이메일 → 스팸 여부 |

**모델 평가 지표:**

| 지표 | 설명 | 범위 | 좋은 값 |
|------|------|------|--------|
| R² (결정계수) | 분산 설명 비율 | 0 ~ 1 | 1에 가까울수록 좋음 |
| MAE | 평균 절대 오차 | 0 ~ ∞ | 0에 가까울수록 좋음 |
| MSE | 평균 제곱 오차 | 0 ~ ∞ | 0에 가까울수록 좋음 |
| RMSE | √MSE (원래 단위) | 0 ~ ∞ | 0에 가까울수록 좋음 |

### 5-2-1. 단순 선형 회귀 (Simple Linear Regression)

$$y = \beta_0 + \beta_1 x + \epsilon$$

- $\beta_0$: 절편 (intercept) — x=0일 때 y값
- $\beta_1$: 기울기 (coefficient) — x가 1 증가할 때 y 변화량
- $\epsilon$: 오차항 (residual)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 단순 선형 회귀: weight → mpg
X = mpg[['weight']]
y = mpg['mpg']

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

print('=== 단순 선형 회귀 결과 ===')
print(f'절편 (β0):  {model.intercept_:.4f}')
print(f'기울기 (β1): {model.coef_[0]:.6f}')
print(f'\n회귀식: mpg = {model.intercept_:.2f} + {model.coef_[0]:.6f} × weight')
print(f'\n=== 모델 평가 ===')
print(f'R²:   {r2_score(y, y_pred):.4f}')
print(f'MAE:  {mean_absolute_error(y, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 회귀선 시각화
sns.regplot(data=mpg, x='weight', y='mpg', ax=axes[0],
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
axes[0].set_title(f'weight → mpg (R²={r2_score(y, y_pred):.3f})')

# 잔차 플롯 (Residual Plot)
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('예측값')
axes[1].set_ylabel('잔차 (Residual)')
axes[1].set_title('잔차 플롯 (패턴 없어야 좋음)')

plt.tight_layout()
plt.show()

### 5-2-2. 다중 선형 회귀 (Multiple Linear Regression)

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n + \epsilon$$

여러 독립변수를 동시에 사용하여 예측 정확도를 높입니다.

**statsmodels OLS**로 p-value까지 확인:
| 출력 항목 | 의미 |
|-----------|------|
| `coef` | 각 변수의 회귀계수 (기울기) |
| `P>|t|` | p-value (0.05 미만이면 유의미) |
| `R-squared` | 모델 설명력 |
| `Adj. R-squared` | 변수 수 보정 R² |
| `[0.025, 0.975]` | 95% 신뢰구간 |

In [ ]:
# 다중 선형 회귀: weight + horsepower + acceleration → mpg
feature_cols = ['weight', 'horsepower', 'acceleration', 'cylinders']
X_multi = mpg[feature_cols]
y = mpg['mpg']

# sklearn
model_multi = LinearRegression()
model_multi.fit(X_multi, y)
y_pred_multi = model_multi.predict(X_multi)

print('=== 다중 선형 회귀 결과 (sklearn) ===')
for col, coef in zip(feature_cols, model_multi.coef_):
    print(f'  {col:<15}: {coef:.4f}')
print(f'  절편:           {model_multi.intercept_:.4f}')
print(f'\nR²:   {r2_score(y, y_pred_multi):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y, y_pred_multi)):.4f}')

In [ ]:
# statsmodels OLS: p-value 포함 상세 결과
X_sm = sm.add_constant(mpg[feature_cols])
ols_model = sm.OLS(y, X_sm).fit()
print(ols_model.summary())

### 5-2-3. 로지스틱 회귀 (Logistic Regression)

종속변수가 **범주형(0/1)**일 때 사용합니다. 확률값(0~1)을 예측합니다.

| 항목 | 선형 회귀 | 로지스틱 회귀 |
|------|-----------|---------------|
| 종속변수 | 연속형 (숫자) | 범주형 (0/1) |
| 출력 | 예측값 | 확률 (sigmoid) |
| 손실 함수 | MSE | Binary Cross-Entropy |
| 평가 | R², RMSE | Accuracy, AUC |

**평가 지표:**

| 지표 | 설명 |
|------|------|
| Accuracy | (TP+TN) / 전체 |
| Precision | TP / (TP+FP) — 양성 예측 중 실제 양성 |
| Recall | TP / (TP+FN) — 실제 양성 중 맞춘 비율 |
| F1-Score | 2 × (Precision × Recall) / (P+R) |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 로지스틱 회귀: 부리 길이/깊이로 성별 예측
peng = penguins[['bill_length_mm', 'bill_depth_mm', 'body_mass_g', 'sex']].copy()
peng['sex_binary'] = (peng['sex'] == 'Male').astype(int)

X_log = peng[['bill_length_mm', 'bill_depth_mm', 'body_mass_g']]
y_log = peng['sex_binary']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

log_model = LogisticRegression()
log_model.fit(X_scaled, y_log)
y_pred_log = log_model.predict(X_scaled)

print('=== 로지스틱 회귀 결과 ===')
print(classification_report(y_log, y_pred_log, target_names=['Female', 'Male']))

In [ ]:
# 혼동 행렬 시각화
cm = confusion_matrix(y_log, y_pred_log)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Female', 'Male'],
            yticklabels=['Female', 'Male'])
plt.title('혼동 행렬 (Confusion Matrix)')
plt.ylabel('실제값')
plt.xlabel('예측값')
plt.show()

## 5-3. 가설 검정 (Hypothesis Testing)

**정의**: 데이터가 특정 가설을 지지하는지 **통계적으로 판단**합니다.

| 개념 | 설명 |
|------|------|
| 귀무가설 (H₀) | 차이가 없다 / 관계가 없다 |
| 대립가설 (H₁) | 차이가 있다 / 관계가 있다 |
| p-value | 귀무가설이 맞다고 가정했을 때 현재 결과가 나올 확률 |
| 유의수준 (α) | 보통 0.05 (5%) 사용 |

**판단**: p-value < 0.05 → 귀무가설 기각 → 통계적으로 유의미한 차이 있음

| 검정 종류 | 함수 | 사용 상황 |
|-----------|------|----------|
| 독립 t-검정 | `stats.ttest_ind()` | 두 독립 그룹 평균 비교 |
| 대응 t-검정 | `stats.ttest_rel()` | 같은 대상의 전/후 비교 |
| 일표본 t-검정 | `stats.ttest_1samp()` | 표본 평균 vs 특정 값 |
| 카이제곱 검정 | `stats.chi2_contingency()` | 두 범주형 변수 독립성 |
| ANOVA | `stats.f_oneway()` | 3개 이상 그룹 평균 비교 |

In [ ]:
# 독립 t-검정: 지역별 연비 차이 (usa vs japan)
mpg_usa    = mpg[mpg['origin'] == 'usa']['mpg']
mpg_japan  = mpg[mpg['origin'] == 'japan']['mpg']

t_stat, p_val = stats.ttest_ind(mpg_usa, mpg_japan)

print('=== 독립 t-검정: USA vs Japan 연비 ===')
print(f'USA   평균: {mpg_usa.mean():.2f}')
print(f'Japan 평균: {mpg_japan.mean():.2f}')
print(f't-통계량: {t_stat:.4f}')
print(f'p-value:  {p_val:.4e}')
if p_val < 0.05:
    print('→ p < 0.05: 두 지역 연비 차이는 통계적으로 유의미함')
else:
    print('→ p >= 0.05: 통계적으로 유의미한 차이 없음')

In [ ]:
# ANOVA: 3개 지역 연비 차이
mpg_europe = mpg[mpg['origin'] == 'europe']['mpg']
f_stat, p_val_anova = stats.f_oneway(mpg_usa, mpg_japan, mpg_europe)

print('=== ANOVA: 3지역 연비 차이 ===')
print(f'F-통계량: {f_stat:.4f}')
print(f'p-value:  {p_val_anova:.4e}')
print('→', '유의미한 차이 있음' if p_val_anova < 0.05 else '차이 없음')

# 카이제곱 검정: 원산지 vs 실린더 수 독립성
ct = pd.crosstab(mpg['origin'], mpg['cylinders'])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f'\n=== 카이제곱 검정: origin vs cylinders ===')
print(f'χ²={chi2:.4f}, p={p_chi:.4e}')
print('→', '두 변수 간 관계 있음' if p_chi < 0.05 else '독립적임')

In [ ]:
# 그룹별 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=mpg, x='origin', y='mpg', ax=axes[0],
            order=['usa', 'europe', 'japan'], palette='Set2')
axes[0].set_title('지역별 연비 분포 (ANOVA 대상)')

sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('원산지 × 실린더 수 교차표 (카이제곱 대상)')

plt.tight_layout()
plt.show()

## 5-4. 군집 분석 (Clustering)

**정의**: 레이블 없이 **비슷한 데이터끼리 그룹화**하는 비지도 학습.

| 알고리즘 | 특징 | 사용 상황 |
|----------|------|----------|
| K-Means | 중심점 기반, 군집 수 k 지정 | 구형 군집, 대용량 데이터 |
| DBSCAN | 밀도 기반, 이상치 탐지 가능 | 비구형 군집, 노이즈 포함 |
| 계층 군집 | 덴드로그램, k 미리 지정 불필요 | 군집 수 모를 때 |

**K-Means 절차:**
1. k개 중심점 무작위 초기화
2. 각 데이터를 가장 가까운 중심에 할당
3. 각 군집의 새 중심 계산
4. 2~3 반복 → 중심 변화 없을 때 종료

**최적 k 선택: Elbow Method**
- WCSS(군집 내 분산) 감소 곡선에서 꺾이는 지점 선택

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 스케일링
cluster_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X_cluster = penguins[cluster_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow Method
wcss = []
K_range = range(1, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, wcss, marker='o', color='steelblue')
plt.xlabel('군집 수 (k)')
plt.ylabel('WCSS')
plt.title('Elbow Method — 최적 k 선택')
plt.xticks(K_range)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# k=3으로 군집화
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
penguins_c = penguins.copy()
penguins_c['cluster'] = km3.fit_predict(X_scaled).astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 군집 결과 산점도
sns.scatterplot(data=penguins_c, x='bill_length_mm', y='body_mass_g',
                hue='cluster', palette='Set1', ax=axes[0], alpha=0.7)
axes[0].set_title('K-Means 군집 결과 (k=3)')

# 실제 종 vs 군집 비교
sns.scatterplot(data=penguins_c, x='bill_length_mm', y='body_mass_g',
                hue='species', palette='Set2', ax=axes[1], alpha=0.7)
axes[1].set_title('실제 종 분류 (비교용)')

plt.tight_layout()
plt.show()

In [ ]:
# 군집별 특성 요약
print('=== 군집별 평균 (K-Means k=3) ===')
print(penguins_c.groupby('cluster')[cluster_cols].mean().round(1))
print('\n=== 군집 vs 실제 종 교차표 ===')
print(pd.crosstab(penguins_c['cluster'], penguins_c['species']))

## 5-5. 주성분 분석 (PCA, Principal Component Analysis)

**정의**: 고차원 데이터를 **중요한 정보는 유지하면서 저차원으로 압축**합니다.

| 개념 | 설명 |
|------|------|
| 주성분 (PC) | 분산을 가장 많이 설명하는 방향의 축 |
| 설명 분산 비율 | 각 PC가 전체 분산에서 차지하는 비율 |
| 누적 설명 분산 | PC1+PC2+... 의 설명력 합계 |

**언제 사용?**
- 시각화: 4차원 이상 데이터를 2D/3D로 표현
- 차원 축소: 모델 학습 전 불필요한 특성 제거
- 다중공선성 제거: 서로 상관된 변수들을 독립 성분으로 변환

**절차**: 스케일링 → PCA 적용 → 설명분산 확인 → 시각화

In [ ]:
from sklearn.decomposition import PCA

# PCA 적용 (전체 4차원 → 분석)
pca = PCA()
pca.fit(X_scaled)

# 설명 분산 비율
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree Plot
axes[0].bar(range(1, len(explained)+1), explained, alpha=0.7, color='steelblue')
axes[0].plot(range(1, len(explained)+1), cumulative, marker='o', color='red', label='누적')
axes[0].axhline(0.9, color='gray', linestyle='--', label='90% 기준선')
axes[0].set_xlabel('주성분 번호')
axes[0].set_ylabel('설명 분산 비율')
axes[0].set_title('Scree Plot')
axes[0].legend()

for i, (e, c) in enumerate(zip(explained, cumulative)):
    axes[0].text(i+1, e+0.01, f'{e:.1%}', ha='center', fontsize=9)

print('설명 분산 비율:')
for i, (e, c) in enumerate(zip(explained, cumulative), 1):
    print(f'  PC{i}: {e:.4f} ({e:.1%}) | 누적: {c:.1%}')

# PCA 2D 시각화
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['species'] = penguins['species'].values

sns.scatterplot(data=pca_df, x='PC1', y='PC2',
                hue='species', palette='Set2', alpha=0.8, ax=axes[1])
axes[1].set_title(
    f'PCA 2D 시각화 (설명력: {pca2.explained_variance_ratio_.sum():.1%})'
)

plt.tight_layout()
plt.show()

In [ ]:
# 주성분 구성 (어떤 변수가 기여하는지)
loadings = pd.DataFrame(
    pca.components_.T,
    index=cluster_cols,
    columns=[f'PC{i+1}' for i in range(len(cluster_cols))]
)
print('=== 주성분 로딩 (각 변수의 기여도) ===')
print(loadings.round(3))

# 로딩 히트맵
plt.figure(figsize=(7, 4))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('주성분 로딩 히트맵')
plt.tight_layout()
plt.show()

## 실습 문제

### 문제 1 — 상관 분석
`tips` 데이터셋(total_bill, tip, size)에서:
1. 상관행렬을 계산하고 히트맵으로 시각화하시오.
2. `total_bill`과 `tip` 간 Pearson 상관계수와 p-value를 출력하시오.

````{admonition} 풀이
:class: dropdown

```python
import seaborn as sns, matplotlib.pyplot as plt
import scipy.stats as stats

tips = sns.load_dataset('tips')

# 1. 상관행렬 + 히트맵
corr = tips[['total_bill', 'tip', 'size']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('tips 상관행렬')
plt.show()

# 2. Pearson 상관계수 + p-value
r, p = stats.pearsonr(tips['total_bill'], tips['tip'])
print(f'r = {r:.4f}, p = {p:.4e}')
print('통계적으로 유의미' if p < 0.05 else '유의미하지 않음')
```
````

### 문제 2 — 회귀 분석
`tips` 데이터셋에서 `total_bill`을 독립변수로 `tip`을 예측하는
단순 선형 회귀 모델을 만들고 R²와 회귀식을 출력하시오.

````{admonition} 풀이
:class: dropdown

```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

X = tips[['total_bill']]
y = tips['tip']

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

print(f'회귀식: tip = {model.intercept_:.4f} + {model.coef_[0]:.4f} × total_bill')
print(f'R²: {r2_score(y, y_pred):.4f}')

# 시각화
import seaborn as sns
sns.regplot(data=tips, x='total_bill', y='tip',
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
plt.title(f'total_bill → tip (R²={r2_score(y, y_pred):.3f})')
plt.show()
```
````

### 문제 3 — 가설 검정
`tips` 데이터셋에서 흡연자(smoker=Yes)와 비흡연자(smoker=No)의
`tip` 금액 평균에 유의미한 차이가 있는지 t-검정으로 확인하시오.

````{admonition} 풀이
:class: dropdown

```python
import scipy.stats as stats

smoker    = tips[tips['smoker'] == 'Yes']['tip']
non_smoker = tips[tips['smoker'] == 'No']['tip']

t, p = stats.ttest_ind(smoker, non_smoker)
print(f'흡연자 평균 tip:    {smoker.mean():.3f}')
print(f'비흡연자 평균 tip:  {non_smoker.mean():.3f}')
print(f't={t:.4f}, p={p:.4f}')
print('차이 있음' if p < 0.05 else '유의미한 차이 없음')
```
````